# 01 — Exploration des données Allego

Ce notebook explore le dataset IRVE filtré sur l'opérateur **Allego**.

**Objectif :** comprendre la structure des données, identifier les valeurs manquantes, et visualiser les distributions clés avant toute modélisation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT / "src"))

PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120


## 1. Chargement & filtrage sur Allego

In [ ]:
RAW_CSV = ROOT / "data" / "raw" / "consolidation-etalab-schema-irve-statique-v-2.3.1-20260505.csv"
df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Dataset complet : {len(df_raw):,} lignes | {len(df_raw.columns)} colonnes")

df = df_raw[df_raw["nom_operateur"] == "Allego"].copy()
print(f"Allego          : {len(df):,} lignes")
print(f"Communes uniques: {df['consolidated_commune'].nunique()}")
print(f"Stations uniques: {df['id_station_itinerance'].nunique()}")


## 2. Aperçu des données

In [ ]:
df.head(3)


In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Colonnes avec valeurs manquantes :")
print(missing)


## 3. Distribution de la puissance nominale

In [ ]:
df["puissance_nominale"] = pd.to_numeric(df["puissance_nominale"], errors="coerce")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["puissance_nominale"].dropna(), bins=40, color="#3498db", edgecolor="white")
ax.set_xlabel("Puissance nominale (kW)")
ax.set_ylabel("Nombre de PDC")
ax.set_title("Distribution de la puissance nominale — Réseau Allego")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_distribution_puissance.png")
plt.show()
print(df["puissance_nominale"].describe())


## 4. Types de connecteurs

In [ ]:
BOOL_COLS = ["prise_type_ef", "prise_type_2", "prise_type_combo_ccs", "prise_type_chademo", "prise_type_autre"]
LABELS    = ["Type EF", "Type 2", "CCS Combo", "CHAdeMO", "Autre"]

for col in BOOL_COLS:
    df[col] = df[col].astype(str).str.lower().map({"true": 1.0, "false": 0.0}).fillna(0.0)

counts = [df[col].sum() for col in BOOL_COLS]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(LABELS, counts, color=sns.color_palette("Set2", len(LABELS)))
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_ylabel("Nombre de PDC")
ax.set_title("Types de connecteurs disponibles — Réseau Allego")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_types_connecteurs.png")
plt.show()


## 5. Types d'implantation

In [ ]:
implantation_counts = df["implantation_station"].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    implantation_counts.values,
    labels=[l[:30] for l in implantation_counts.index],
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2", len(implantation_counts)),
    startangle=140,
)
ax.set_title("Répartition par type d'implantation — Réseau Allego")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_implantation.png")
plt.show()


## 6. Condition d'accès

In [ ]:
acces_counts = df["condition_acces"].value_counts()
print(acces_counts)

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(acces_counts.index, acces_counts.values, color=["#2ecc71", "#e74c3c"])
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_xlabel("Nombre de PDC")
ax.set_title("Condition d'accès — Réseau Allego")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_condition_acces.png")
plt.show()


## 7. Carte géographique des bornes Allego

In [ ]:
df["latitude"]  = pd.to_numeric(df["consolidated_latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["consolidated_longitude"], errors="coerce")
df_geo = df.dropna(subset=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(df_geo["longitude"], df_geo["latitude"], alpha=0.4, s=8, color="#3498db")
ax.set_xlim(-5.5, 10.5)
ax.set_ylim(41, 52)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Localisation des bornes Allego en France")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_carte_allego.png")
plt.show()
print(f"PDC localisés : {len(df_geo):,}")


---
**Fin du notebook 01.** → Passer au notebook `02_clustering.ipynb`